# Topic: SQL Median Calculation

## Definition (30-second explanation)
* The median is the precise middle value of a dataset sorted in ascending or descending order.
* If the dataset has an odd number of values, it is the exact middle row. 
* If the dataset has an even number of values, the median is the average of the two middle rows.
* Most SQL dialects lack a built-in `MEDIAN()` aggregate function, making it a test of advanced query skills.

## Why Interviewers Ask This
* It is one of the most popular SQL interview questions at top companies like Amazon, Google, Meta, and Stripe.
* Tests your mastery of window functions (like `ROW_NUMBER()`) and self-joins or CTEs.
* Evaluates your attention to edge cases like even/odd row counts, integer division truncation, and `NULL` handling.

## Core Concepts
* **`PERCENTILE_CONT(0.5)`**: A clean, readable window function available in PostgreSQL, SQL Server, Oracle, and Snowflake that automatically handles interpolation.
* **Universal `ROW_NUMBER()` Method**: Using ascending and descending row numbers to logically isolate the middle row(s) in dialects like MySQL.
* **The `ABS` Trick**: Filtering where `ABS(rn_asc - rn_desc) <= 1` elegantly captures both odd (difference of 0) and even (difference of 1) middle rows.

## When to Use
* Essential for analyzing skewed distributions where outliers distort the mean.
* Standard metric for income/salary distribution, real estate pricing, and customer lifetime value.

## Advantages
* Highly robust to extreme outliers (e.g., one CEO salary won't skew the metric for standard employees).
* Provides a more accurate reflection of the "typical" or "average" user/case in skewed data.

## Limitations
* Computationally expensive because calculating the median always requires fully sorting the dataset.
* Cumbersome syntax in standard SQL compared to simple aggregations like `AVG()` or `SUM()`.

## Common Comparisons
* **Mean vs. Median**: Mean (`AVG()`) is heavily influenced by outliers, while Median is robust.
* **`PERCENTILE_CONT` vs. `PERCENTILE_DISC`**: `CONT` mathematically interpolates the exact median for even datasets, whereas `DISC` simply returns an existing actual data point from the set.

## Common Interview Traps
* **Ignoring NULLs**: NULLs can shift row counts; always use `WHERE column IS NOT NULL`.
* **Integer Truncation**: In some dialects, dividing an integer count by 2 truncates the decimal; use `total/2.0` or cast to float.
* **Ignoring Grouping Context**: Failing to clarify if the interviewer wants the median for the entire dataset or grouped by a specific category.

## Python / SQL Syntax
**Method 1: Modern SQL (PostgreSQL, Snowflake, etc.)**
```sql
SELECT department,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary) AS median_salary
FROM employees
GROUP BY department;
```

**Method 2: Universal Approach (MySQL, etc.)**
```sql
WITH numbered AS (
  SELECT department, salary,
         ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary ASC) AS rn_asc,
         ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn_desc
  FROM employees
  WHERE salary IS NOT NULL
)
SELECT department, AVG(salary) AS median_salary
FROM numbered
WHERE ABS(rn_asc - rn_desc) <= 1
GROUP BY department;
```

## 45-Second Interview Answer
"Because standard SQL doesn't have a built-in MEDIAN() function, I would calculate it based on the database engine. If we are using PostgreSQL or Snowflake, I would use the PERCENTILE_CONT(0.5) WITHIN GROUP function because it cleanly handles even and odd counts. If I need a universal solution for something like MySQL, I would use a CTE to assign ascending and descending ROW_NUMBER()s partitioned by the group, filter out NULL values, and then average the rows where the absolute difference between the ascending and descending row numbers is less than or equal to 1."